# 6. Symbolic regression

Imagine you want to figure out the formula speed of sound in ideal gases.

In your lab (or in our case the FEM software), you have the following gases available: Air, $\mathrm{N_2}$, $\mathrm{CO_2}$, $\mathrm{O_2}$, $\mathrm{H_2}$, and $\mathrm{He}$
So you take a Kundt tube open on both sides with length $L$ = 1 m and measure the fundamental frequency across varying temperatures $T$.
Then, using the open-open resonance condition $c = f_0 \lambda = f_0 2L$, you obtain the speed of sound. 

What now? In the times of Johannes Kepler, you would embark on a journey of fitting your table of numbers and after a few years, we would meet to share the results.
Today, this is the core task of symbolic regression.

API reference for Python Symbolic Regression (PySR) can be found [here](https://ai.damtp.cam.ac.uk/pysr/v1.5.9/api), if you want to play around with more settings.

In [ ]:
# import necessary libraries
import numpy as np
import matplotlib.pyplot as plt

from pysr import PySRRegressor # importing this might take a few min the first time you run it


In [ ]:
# --- 1. Load Data ---
data = np.loadtxt("sr_data.csv", delimiter=",", skiprows=1, dtype=np.float64)

# Unpack relevant columns
mat_idx = data[:, 0].astype(int)
T_var   = data[:, 1]  # Temperature [K]
freq    = data[:, 2]  # Frequency [Hz]
rho     = data[:, 3]  # Density [kg/m^3]
gamma   = data[:, 4]  # Heat capacity ratio [1]
Mn      = data[:, 5]  # Molar mass [kg/mol]
R_const = data[:, 6]  # Universal gas constant [J/(mol*K)]

L = 1.0  # Length of the tube in meters
c = freq * 2 * L


In [ ]:
# Target vector y
y = c

# --- 3. Feature Matrix X for PySR ---
# Features: [temperature, density, gamma, Mn, R_const]
X = np.column_stack((T_var, rho, gamma, Mn, R_const))

# Variable names for PySR configuration
feature_names = ["T", "rho", "g", "Mn", "R"]

X_units = ["K", "kg/m^3", "1", "kg/mol", "J/(mol*K)"]
y_units = "m/s"

print(f"X shape: {X.shape} (Features: {feature_names})")
print(f"y shape: {y.shape} (Target: Speed of sound c in m/s)")


In [ ]:
model = PySRRegressor(
    niterations=100, 
    populations=15,
    population_size=50,
    model_selection="best",
    binary_operators=["*", "+", "-","/"],
    unary_operators=[
        "sqrt",
        # "square",
        # "sin", 
        # "cos",
        # "exp"
    ],
    elementwise_loss='L1DistLoss()',
    maxsize = 20,
    maxdepth=8,
)

model.fit(X, y, variable_names=feature_names)

print(model.sympy())
print(model.latex(precision=3))



In [ ]:
# Compute predictions and relative error
ypredict = model.predict(X)
rel_error_pct = ((ypredict - y) / y) * 100
sample_order = np.arange(len(y))

plt.figure(figsize=(16, 4.5))

# 1. Dataset Order vs Speed of Sound (Truth vs Prediction)
plt.subplot(1, 3, 1)
plt.plot(sample_order, y, 'o', color='tab:blue', label='Ground Truth', alpha=0.7)
plt.plot(sample_order, ypredict, '+', color='tab:red', label='Prediction', alpha=0.9, markersize=8)
plt.title('Speed of Sound per Sample')
plt.xlabel('Sample index / 1')
plt.ylabel('c / (m/s)')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='best')

# 2. Relative Error (%)
plt.subplot(1, 3, 2)
plt.plot(sample_order, rel_error_pct, 's', color='tab:purple', alpha=0.8)
plt.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.7)
plt.title('Relative Error')
plt.xlabel('Sample index / 1')
plt.ylabel('Relative Error / %')
plt.grid(True, linestyle=':', alpha=0.6)

# 3. Parity Plot: Truth vs Prediction
plt.subplot(1, 3, 3)
min_val = min(np.min(y), np.min(ypredict)) * 0.95
max_val = max(np.max(y), np.max(ypredict)) * 1.05
ref_line = np.linspace(min_val, max_val, 100)

plt.plot(y, ypredict, 'x', color='tab:blue',label='Samples')
plt.plot(ref_line, ref_line, 'r--', label='Ideal (1:1)')
plt.title('Truth vs Prediction')
plt.xlabel('c_true / (m/s)')
plt.ylabel('c_pred / (m/s)')
plt.xlim(min_val, max_val)
plt.ylim(min_val, max_val)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper left')

plt.tight_layout()
plt.show()

Note that we completely neglected any units until now.
Let's refit it with dimensional constraints (more details [here](https://ai.damtp.cam.ac.uk/pysr/v1.5.9/examples#_10-dimensional-constraints)).

In [ ]:
model2 = PySRRegressor(
    niterations=100, 
    populations=15,
    population_size=50,
    model_selection="best",
    binary_operators=["*", "+", "-","/"],
    unary_operators=[
        "sqrt",
        # "square",
        # "sin", 
        # "cos",
        # "exp"
    ],
    elementwise_loss='L1DistLoss()',
    maxsize = 20,
    # maxdepth=8,
    dimensional_constraint_penalty=1000,
    dimensionless_constants_only=True
)

model2.fit(X, y, 
          variable_names=feature_names,
          X_units = X_units,
          y_units = y_units
          )

print(model2.sympy())
print(model2.latex(precision=3))



In [ ]:
# Compute predictions and relative error
ypredict = model2.predict(X)
rel_error_pct = ((ypredict - y) / y) * 100
sample_order = np.arange(len(y))

plt.figure(figsize=(16, 4.5))

# 1. Dataset Order vs Speed of Sound (Truth vs Prediction)
plt.subplot(1, 3, 1)
plt.plot(sample_order, y, 'o', color='tab:blue', label='Ground Truth', alpha=0.7)
plt.plot(sample_order, ypredict, '+', color='tab:red', label='Prediction', alpha=0.9, markersize=8)
plt.title('Speed of Sound per Sample')
plt.xlabel('Sample index / 1')
plt.ylabel('c / (m/s)')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='best')

# 2. Relative Error (%)
plt.subplot(1, 3, 2)
plt.plot(sample_order, rel_error_pct, 's', color='tab:purple', alpha=0.8)
plt.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.7)
plt.title('Relative Error')
plt.xlabel('Sample index / 1')
plt.ylabel('Relative Error / %')
plt.grid(True, linestyle=':', alpha=0.6)

# 3. Parity Plot: Truth vs Prediction
plt.subplot(1, 3, 3)
min_val = min(np.min(y), np.min(ypredict)) * 0.95
max_val = max(np.max(y), np.max(ypredict)) * 1.05
ref_line = np.linspace(min_val, max_val, 100)

plt.plot(y, ypredict, 'x', color='tab:blue',label='Samples')
plt.plot(ref_line, ref_line, 'r--', label='Ideal (1:1)')
plt.title('Truth vs Prediction')
plt.xlabel('c_true / (m/s)')
plt.ylabel('c_pred / (m/s)')
plt.xlim(min_val, max_val)
plt.ylim(min_val, max_val)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper left')

plt.tight_layout()
plt.show()